# **ISOM4030 Lab 10 - Sequence-to-Sequence Models for Machine Translation**

## Part 10: Language Translation using Encoder-Decoder (Seq2seq) Architecture

In this lab, we will build a Neural Machine Translation (NMT) system that translates sentences from Chinese to English. We will use an Encoder-Decoder architecture implemented with Recurrent Neural Networks (RNNs) using GRU cells.

You will learn how to:
- Preprocess and standardize multilingual text data (English and Chinese).
- Tokenize text sequences and build vocabulary using Keras' `TextVectorization` layer.
- Implement an **Encoder** and a **Decoder** using Keras Model Subclassing.
- Train the Sequence-to-Sequence (Seq2seq) model with a custom masked loss and accuracy.
- Build an interactive translation tool.


# **Step 0: Setup**


In [1]:
import keras
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Embedding, RNN, GRUCell, GRU, Dense, Flatten, Layer, Concatenate
from tensorflow.keras.layers import TextVectorization,  StringLookup
from tensorflow.keras import Sequential, Input, Model, callbacks, losses, optimizers

import time, os, re

import numpy as np
import matplotlib.pyplot as plt

In [2]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))


Num GPUs Available:  1


# **Step 1: Load and Split the Data**

We will use an English-Chinese translation dataset provided by [ManyThings.org](http://www.manythings.org/anki/). This dataset contains tab-separated sentence pairs, for example:
```
He called me a cab.    他替我叫了一部計程車。
```


In [3]:
# download the file and return the absolute path as a string
path_to_file = tf.keras.utils.get_file('eng-chinese.txt', origin="https://raw.githubusercontent.com/justinjiajia/datafiles/master/eng-chinese.txt",
                                       cache_dir=".", cache_subdir="")
print("\nThe path to the downloaded file is:", path_to_file)

1269203/1269203 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

The path to the downloaded file is: ./eng-chinese.txt


In [4]:
with open(path_to_file, "r") as f:
    # read all the text, and split it line by line
    list_of_lines = f.read().splitlines()

print(f"Total sentence pairs: {len(list_of_lines)}")

Total sentence pairs: 20289


We divide the tab-separated paired sentences into two arrays: `context_sentences` (the source language: Chinese) and `target_sentences` (the target language: English).


In [5]:
pairs = [line.split('\t') for line in list_of_lines]

context_sentences = np.array([context for target, context in pairs])
target_sentences = np.array([target for target, context in pairs])

Let's print the first 5 pairs to see how the sentences match up:


In [6]:
for i in range(5):
    print(f"Pair {i+1}:")
    print(f"  Source (Chi): {context_sentences[i]}")
    print(f"  Target (Eng): {target_sentences[i]}")

Pair 1:
  Source (Chi): 嗨。
  Target (Eng): Hi.
Pair 2:
  Source (Chi): 你好。
  Target (Eng): Hi.
Pair 3:
  Source (Chi): 你用跑的。
  Target (Eng): Run.
Pair 4:
  Source (Chi): 等等！
  Target (Eng): Wait!
Pair 5:
  Source (Chi): 你好。
  Target (Eng): Hello!


And here are the last 5 pairs:


In [7]:
for i in range(-5, 0):
    print(f"Pair {len(context_sentences)+i+1}:")
    print(f"  Source (Chi): {context_sentences[i]}")
    print(f"  Target (Eng): {target_sentences[i]}")

Pair 20285:
  Source (Chi): 去年在菲律宾，地震和海啸造成了超过6000人的死亡。
  Target (Eng): Last year in the Philippines, earthquakes and tidal waves resulted in the deaths of more than 6,000 people.
Pair 20286:
  Source (Chi): 我母亲的法语比我父亲的英语要好，所以他们通常用法语交流。
  Target (Eng): My mother speaks French better than my father speaks English, so they usually speak to each other in French.
Pair 20287:
  Source (Chi): 汤姆不知如何翻译“计算机”一词，因为同他谈话的人从未见过一台。
  Target (Eng): Tom didn't know how to translate the word "computer" because the people he was talking to had never seen one.
Pair 20288:
  Source (Chi): 即使是现在，我偶尔还是想见到你。不是今天的你，而是我记忆中曾经的你。
  Target (Eng): Even now, I occasionally think I'd like to see you. Not the you that you are today, but the you I remember from the past.
Pair 20289:
  Source (Chi): 如果一個人在成人前沒有機會習得目標語言，他對該語言的認識達到母語者程度的機會是相當小的。
  Target (Eng): If a person has not had a chance to acquire his target language by the time he's an adult, he's unlikely to be able to reach native speaker level in that language.

Next, we split these pairs of sentences into training (90%) and validation (10%) sets, and create a `tf.data.Dataset` of strings in order to shuffle and batch them efficiently.


In [8]:
BUFFER_SIZE = len(list_of_lines)
BATCH_SIZE = 64

# a 90-10 split
is_train = np.random.uniform(size=(len(list_of_lines),)) < 0.9

train_text = tf.data.Dataset.from_tensor_slices((
                      np.array(context_sentences)[is_train],
                      np.array(target_sentences)[is_train]
                  )).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)

val_text = tf.data.Dataset.from_tensor_slices((
                      np.array(context_sentences)[~is_train],
                      np.array(target_sentences)[~is_train]
                  )).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)

Let's take one batch to inspect the number of examples and the structure of our tensor elements:


In [9]:
for example_context_strings, example_target_strings in train_text.take(1):
    print(example_context_strings[:5])
    print()
    print(example_target_strings[:5])


tf.Tensor(
[b'\xe6\x88\x91\xe5\x85\x8d\xe8\xb2\xbb\xe5\xbe\x97\xe5\x88\xb0\xe9\x80\x99\xe5\x80\x8bCD\xe6\x92\xad\xe6\x94\xbe\xe6\xa9\x9f\xe3\x80\x82'
 b'\xe4\xbd\xa0\xe5\x87\x8f\xe8\x82\xa5\xe4\xba\x86\xef\xbc\x8c\xe4\xb8\x8d\xe6\x98\xaf\xe4\xb9\x88\xef\xbc\x9f'
 b'\xe9\xa3\x8e\xe9\x99\xa9\xe5\xa4\xaa\xe5\xa4\xa7\xe3\x80\x82'
 b'\xe6\xb9\xaf\xe5\xa7\x86\xe4\xb8\x8d\xe8\x80\x81\xe3\x80\x82'
 b'\xe4\xbd\xa0\xe6\x9c\x83\xe9\x96\x8b\xe8\xbb\x8a\xef\xbc\x8c\xe4\xb8\x8d\xe6\x98\xaf\xe5\x97\x8e?'], shape=(5,), dtype=string)

tf.Tensor(
[b'I got this CD player for free.' b"You've lost weight, haven't you?"
 b'The risk is too great.' b"Tom isn't old."
 b"You can drive a car, can't you?"], shape=(5,), dtype=string)


# **Step 2: Preprocess and Standardize the Texts**

To prepare our text data for training, we need to apply standardization steps:
1. **Lowercase all characters** to reduce vocabulary size.
2. **Remove special characters** while retaining letters, numbers, and basic punctuation.
3. **Add spaces around punctuation** so they are tokenized as individual words.
4. **Prepend a `[START]` token and append an `[END]` token** to each sentence.

In English, words are typically separated by spaces, making tokenization straightforward. In contrast, Chinese lacks clear delimiters between words. Thus, we define two separate preprocessing functions for English and Chinese, respectively.


In [10]:
def eng_preprocess(text):

  # lowercase all characters
  text = tf.strings.lower(text)
  # keep space, a to z, and 0 to 9, and select punctuation
  text = tf.strings.regex_replace(text, '[^ a-z0-9.?!,"\']', ' ')
  # remove comma separators in numbers
  text =  tf.strings.regex_replace(text, r'(\d{1,3}),', r'\1')
  # add spaces around punctuation and numbers
  text = tf.strings.regex_replace(text, '[.?!,"\'0-9]', r' \0 ')
  # replace multiple spaces with one space
  text = tf.strings.regex_replace(text, '[ ]+', ' ')
  # strip whitespace
  text = tf.strings.strip(text)
  # prepend the start token and append the end token
  text = tf.strings.join(['[START]', text, '[END]'], separator=' ')

  return text

In [11]:
def chi_preprocess(text):

  # some sentences also contain English words
  text = tf.strings.lower(text)
  # add the space between non-English characters
  text = tf.strings.regex_replace(text, '[^a-z]', r' \0 ')
  # replace multiple spaces with one space
  text = tf.strings.regex_replace(text, '[ ]+', ' ')
  # replace several spaces with one space
  text = tf.strings.strip(text)
  # prepend the start token and append the end token
  text = tf.strings.join(['[START]', text, '[END]'], separator=' ')

  return text

Run the cells below to see the effects of the preprocessing functions. Note that TensorFlow's `tf.string` tensor uses **byte strings** by default (indicated by the prefix `b`). Refer to this [webpage](https://realpython.com/python-encodings-guide/) if you want to know more about how Python encodes multilingual characters and even emojis.


In [12]:
eng_preprocess(["Tom didn't see Mary's cat."])


<tf.Tensor: shape=(1,), dtype=string, numpy=array([b"[START] tom didn ' t see mary ' s cat . [END]"], dtype=object)>

In [13]:
chi_preprocess(["這個年輕人比他哥哥更誠實。"])


<tf.Tensor: shape=(1,), dtype=string, numpy=
array([b'[START] \xe9\x80\x99 \xe5\x80\x8b \xe5\xb9\xb4 \xe8\xbc\x95 \xe4\xba\xba \xe6\xaf\x94 \xe4\xbb\x96 \xe5\x93\xa5 \xe5\x93\xa5 \xe6\x9b\xb4 \xe8\xaa\xa0 \xe5\xaf\xa6 \xe3\x80\x82 [END]'],
      dtype=object)>

The following code decodes the byte strings into human-readable, standard Python strings:


In [14]:
[example_str.decode() for example_str in eng_preprocess(["Tom didn't see Mary's cat."]).numpy()]


["[START] tom didn ' t see mary ' s cat . [END]"]

In [15]:
[example_str.decode() for example_str in chi_preprocess(["這個年輕人比他哥哥更誠實。"]).numpy()]


['[START] 這 個 年 輕 人 比 他 哥 哥 更 誠 實 。 [END]']

### **Reflection Question 10.1**

(a) Why do we need different preprocessing functions for English and Chinese text? (Hint: Consider word boundaries and space delimiters in both languages).

(b) What is the purpose of adding the `[START]` and `[END]` tokens to our sentences? Why is this particularly important for the Decoder during sequence generation?

**Answer:**
*Write your answer here (1-3 sentences per part).*


# **Step 3: Build the Vocabularies**

Next, we will use Keras' `TextVectorization` layer to extract vocabulary and convert sentences to token ID sequences.
We set `ragged=True` so that the vectorizer outputs ragged tensors, allowing us to defer the uniform padding step until a later stage.


In [ ]:
context_text_processor = TextVectorization(
    standardize=___________________,
    ragged=_________________
)


In [ ]:
# define a custom function to extract only the source sentence from each example
def remove_target(context, target):
    return context


# .adapt() reads the training data, and determines the vocabulary
context_text_processor.adapt(train_text.map(remove_target))


# get the vocabulary built by this layer
context_vocab = context_text_processor.________________________

# here are the first 10 words from the vocabulary:
context_vocab[:10]

['',
 '[UNK]',
 np.str_('[START]'),
 np.str_('[END]'),
 np.str_('。'),
 np.str_('我'),
 np.str_('的'),
 np.str_('了'),
 np.str_('你'),
 np.str_('他')]

The layer can convert a batch of Chinese strings into a batch of token IDs:


In [19]:
context_tokens_batch = context_text_processor(example_context_strings[:5])
print(context_tokens_batch)


<tf.RaggedTensor [[2, 5, 724, 596, 46, 26, 18, 25, 1525, 1276, 220, 244, 4, 3],
 [2, 8, 1626, 1387, 7, 22, 10, 11, 44, 14, 3],
 [2, 779, 781, 145, 59, 4, 3], [2, 40, 16, 10, 178, 4, 3],
 [2, 8, 43, 91, 75, 22, 10, 11, 32, 70, 3]]>


Since there is no default method in `TextVectorization` to convert index IDs back to words, we use Keras `StringLookup` with `invert=True` to create an inverse lookup table:


In [ ]:
chi_idx_to_word = StringLookup(
    vocabulary=_______________________________,
    mask_token=____________,
    invert=_______________
)


In [21]:
chi_idx_to_word(context_tokens_batch)


<tf.RaggedTensor [[b'[START]', b'\xe6\x88\x91', b'\xe5\x85\x8d', b'\xe8\xb2\xbb',
  b'\xe5\xbe\x97', b'\xe5\x88\xb0', b'\xe9\x80\x99', b'\xe5\x80\x8b', b'cd',
  b'\xe6\x92\xad', b'\xe6\x94\xbe', b'\xe6\xa9\x9f', b'\xe3\x80\x82',
  b'[END]']                                                                 ,
 [b'[START]', b'\xe4\xbd\xa0', b'\xe5\x87\x8f', b'\xe8\x82\xa5',
  b'\xe4\xba\x86', b'\xef\xbc\x8c', b'\xe4\xb8\x8d', b'\xe6\x98\xaf',
  b'\xe4\xb9\x88', b'\xef\xbc\x9f', b'[END]']                        ,
 [b'[START]', b'\xe9\xa3\x8e', b'\xe9\x99\xa9', b'\xe5\xa4\xaa',
  b'\xe5\xa4\xa7', b'\xe3\x80\x82', b'[END]']                   ,
 [b'[START]', b'\xe6\xb9\xaf', b'\xe5\xa7\x86', b'\xe4\xb8\x8d',
  b'\xe8\x80\x81', b'\xe3\x80\x82', b'[END]']                   ,
 [b'[START]', b'\xe4\xbd\xa0', b'\xe6\x9c\x83', b'\xe9\x96\x8b',
  b'\xe8\xbb\x8a', b'\xef\xbc\x8c', b'\xe4\xb8\x8d', b'\xe6\x98\xaf',
  b'\xe5\x97\x8e', b'?', b'[END]']                                   ]>

### **Exercise: Build the English Vectorizer and Lookup Table**

Now, build and adapt the English `TextVectorization` layer and its corresponding inverse lookup table `eng_idx_to_word`.


In [ ]:
target_text_processor = TextVectorization(
    standardize=_____________________,
    ragged=______________
)
target_text_processor.adapt(np.array(target_sentences)[is_train])


Verify that the English text processor and inverse lookup function work correctly:


In [23]:
target_tokens_batch = target_text_processor(example_target_strings[:5])
print(target_tokens_batch)


<tf.RaggedTensor [[2, 5, 91, 25, 1614, 982, 29, 395, 4, 3],
 [2, 9, 6, 76, 292, 660, 26, 337, 6, 14, 9, 11, 3],
 [2, 7, 1372, 12, 130, 374, 4, 3], [2, 15, 161, 6, 14, 145, 4, 3],
 [2, 9, 32, 313, 10, 103, 26, 32, 6, 14, 9, 11, 3]]>


In [ ]:
eng_idx_to_word = StringLookup(
    vocabulary=target_text_processor._______________________,
    mask_token=_________,
    invert=____________
)
eng_idx_to_word(target_tokens_batch)


<tf.RaggedTensor [[b'[START]', b'i', b'got', b'this', b'cd', b'player', b'for', b'free',
  b'.', b'[END]']                                                      ,
 [b'[START]', b'you', b"'", b've', b'lost', b'weight', b',', b'haven', b"'",
  b't', b'you', b'?', b'[END]']                                             ,
 [b'[START]', b'the', b'risk', b'is', b'too', b'great', b'.', b'[END]'],
 [b'[START]', b'tom', b'isn', b"'", b't', b'old', b'.', b'[END]'],
 [b'[START]', b'you', b'can', b'drive', b'a', b'car', b',', b'can', b"'",
  b't', b'you', b'?', b'[END]']                                          ]>

Here are the vocabulary sizes of our source and target languages:


In [25]:
print(f"Vocabulary size of the source language: {len(context_text_processor.get_vocabulary())}")
print(f"Vocabulary size of the target language: {len(target_text_processor.get_vocabulary())}")


Vocabulary size of the source language: 3341
Vocabulary size of the target language: 5852


# **Step 4: Tokenize and Batch the Dataset**

Currently, our sequences have varying lengths. To feed them to our RNN in uniform batches, we convert ragged tensors into dense tensors using `.to_tensor()`, which automatically zero-pads each sequence to match the length of the longest sequence in the batch.


In [26]:
print(context_tokens_batch.to_tensor().shape)


(5, 14)


Next, we define `process_text` to:
1. Convert the `Datasets` of strings into zero-padded tensors of token IDs.
2. Split each English target sequence into `target_in` (for teacher forcing, ending without `[END]`) and `target_out` (shifted by one step, representing the expected next-token labels).


In [27]:
def process_text(context, target):
    context = context_text_processor(context).to_tensor()
    target = target_text_processor(target)
    t_in = target[:, :-1].to_tensor()
    t_out = target[:, 1:].to_tensor()
    return (context, t_in), t_out

train_ds = train_text.map(process_text, tf.data.AUTOTUNE)
val_ds = val_text.map(process_text, tf.data.AUTOTUNE)


In [28]:
for (context, target_in), target_out in train_ds.take(4):
    print(f"Context shape:    {context.shape}")
    print(f"Target in shape:  {target_in.shape}")
    print(f"Target out shape: {target_out.shape}")
    print('-' * 40)


Context shape:    (64, 31)
Target in shape:  (64, 19)
Target out shape: (64, 19)
----------------------------------------
Context shape:    (64, 22)
Target in shape:  (64, 22)
Target out shape: (64, 22)
----------------------------------------
Context shape:    (64, 19)
Target in shape:  (64, 13)
Target out shape: (64, 13)
----------------------------------------
Context shape:    (64, 20)
Target in shape:  (64, 15)
Target out shape: (64, 15)
----------------------------------------


In [29]:
print("The first 4 context sentences in the first batch:")
print(chi_idx_to_word(context[:4]))
print("\nThe target inputs:")
print(eng_idx_to_word(target_in[:4]))
print("\nThe target outputs (labels to predict):")
print(eng_idx_to_word(target_out[:4]))


The first 4 context sentences in the first batch:
tf.Tensor(
[[b'[START]' b'\xe5\xa5\xb9' b'\xe4\xb8\x8d' b'\xe5\x96\x9c'
  b'\xe6\xac\xa2' b'\xe5\x94\xb1' b'\xe6\x82\xb2' b'\xe4\xbc\xa4'
  b'\xe7\x9a\x84' b'\xe6\xad\x8c' b'\xe3\x80\x82' b'[END]' b'' b'' b''
  b'' b'' b'' b'' b'']
 [b'[START]' b'\xe6\x88\x91' b'\xe6\x9c\x83' b'\xe6\xb8\xb8'
  b'\xe6\xb3\xb3' b'\xe3\x80\x82' b'[END]' b'' b'' b'' b'' b'' b'' b''
  b'' b'' b'' b'' b'' b'']
 [b'[START]' b'\xe5\xa5\xb9' b'\xe5\x96\x9c' b'\xe6\xad\xa1'
  b'\xe6\x9f\xb3' b'\xe6\xa9\x99' b'\xe3\x80\x82' b'[END]' b'' b'' b''
  b'' b'' b'' b'' b'' b'' b'' b'' b'']
 [b'[START]' b'\xe9\x82\xa3' b'\xe5\x80\x8b' b'\xe5\xa5\xb3'
  b'\xe5\xad\xa9' b'\xe9\x95\xb7' b'\xe5\xbe\x97' b'\xe5\xbe\x88'
  b'\xe5\xa5\xbd' b'\xe7\x9c\x8b' b'\xe3\x80\x82' b'[END]' b'' b'' b''
  b'' b'' b'' b'' b'']], shape=(4, 20), dtype=string)

The target inputs:
tf.Tensor(
[[b'[START]' b'she' b'doesn' b"'" b't' b'like' b'to' b'sing' b'sad'
  b'songs' b'.' b'' b'' b'' b'']
 [b'

# **Step 5: Build a Seq2seq MT Model**

The Sequence-to-Sequence (Seq2seq) model is based on the **Encoder-Decoder Architecture**:
- **Encoder**: Converts the input Chinese sentence into a high-dimensional hidden state vector that summarizes the source text.
- **Decoder**: Receives the Encoder's state and generates the corresponding English sentence token-by-token.

Both are implemented using Gated Recurrent Units (GRUs) to model temporal dependencies.


In [30]:
embedding_dim = 256
units = 1024
num_layers = 2

### **5.1 The Seq2Seq Encoder**
The Encoder maps the input token IDs to dense embeddings, then processes them through multi-layer GRUs.


In [31]:
# use Model as the base class because we want to use .summary() later
# https://www.tensorflow.org/guide/keras/making_new_layers_and_models_via_subclassing#the_model_class
class Encoder(Model):

    # the initializer method
    def __init__(self, text_processor, embedding_dim, encoder_units, num_layers, **kwargs):
        super(Encoder, self).__init__(**kwargs)
        self.text_processor = text_processor
        self.vocab_size = text_processor.vocabulary_size()
        self.embedding = Embedding(self.vocab_size, embedding_dim, mask_zero=True)
        self.rnn = RNN([GRUCell(encoder_units, dropout=0.2) for _ in range(num_layers)],
                        return_state=True)

    # the call method that defines all computations required for a forward pass; it uses the functional API
    def call(self, input):
        x = self.embedding(input)
        # what is output by this multi-layer RNN is a Python tuple
        # the first element (i.e., rnn_output[0]) is a 2-dimensional array that packs the RNN block's last-step outputs across batch examples.
        # the remaining elements (i.e., rnn_output[1:]) are the last-step hidden states from different layers.
        # the hidden state of the bottom layer comes first, and is followed by those of the subsequent layers in order.
        rnn_output = self.rnn(x)
        # rnn_output is a tuple of 3 elements; the latter two are each a list of a single 2-dimensional array
        # (array<256, 1024>,  [array<256, 1024>],  [array<256, 1024>])
        return rnn_output[0], rnn_output[1:]

In [32]:
# create an Encoder instance
encoder = Encoder(context_text_processor, embedding_dim, units, num_layers)

# pass the first batch through it to check the dimensionality of the intermediate output
encoder_output, encoder_state = encoder(context)

print(f'Encoder is a multi-layer RNN with {len(encoder_state)} layers\n')
print(f'The shape of Encoder\'s last-step output is: {encoder_output.shape}\n')

for idx, state_by_layer in enumerate(encoder_state):
    print(f'the shape of Encoder\'s last-step state at layer {idx+1} is: {state_by_layer[0].shape}')

Encoder is a multi-layer RNN with 2 layers

The shape of Encoder's last-step output is: (64, 1024)

the shape of Encoder's last-step state at layer 1 is: (64, 1024)
the shape of Encoder's last-step state at layer 2 is: (64, 1024)


### **5.2 The Seq2Seq Decoder**
The Decoder is initialized with the Encoder's final hidden state. It predicts the next token in the target sequence at each step.


In [33]:
class Decoder(Model):

    def __init__(self, text_processor, idx_to_word,
                 embedding_dim, decoder_units, num_layers, **kwargs):
        super(Decoder, self).__init__(**kwargs)
        self.text_processor = text_processor
        self.vocab_size = text_processor.vocabulary_size()
        self.idx_to_word = idx_to_word
        self.embedding = Embedding(self.vocab_size, embedding_dim, mask_zero=True)
        self.rnn = RNN([GRUCell(decoder_units, dropout=0.2) for _ in range(num_layers)],
                        return_sequences=True, return_state=True)
        self.dense = Dense(self.vocab_size)
        self.concat = Concatenate()

    def call(self, context, input, state=None):

        x = self.embedding(input)
        context = tf.repeat(tf.expand_dims(context, axis=1), repeats=tf.shape(x)[1], axis=1)
        x_and_context = self.concat([x, context])
        # the output of the RNN block is again a Python tuple
        # the first element is a 3-dimensional tensor that packs the RNN block's all-step outputs across batch examples.
        rnn_output = self.rnn(x_and_context, initial_state=state)
        output = self.dense(rnn_output[0])

        return output, rnn_output[1:]

In [34]:
# create an Decoder instance
decoder = Decoder(target_text_processor, eng_idx_to_word, embedding_dim, units, num_layers)

# pass the first batch through the encoder and decorder in order to check the dimensionality of the model output
encoder_output, encoder_state = encoder(context)
decoder_output, decoder_state = decoder(encoder_state[-1][0], target_in, encoder_state)

print(f'Decoder is a multi-layer RNN with {len(decoder_state)} layers\n')
print(f'The shape of Decoder\'s multi-step output is: {decoder_output.shape}\n')

for idx, state_by_layer in enumerate(decoder_state):
    print(f'The shape of Decoder\'s last-step state at layer {idx+1} is: {state_by_layer[0].shape}')

Decoder is a multi-layer RNN with 2 layers

The shape of Decoder's multi-step output is: (64, 15, 5852)

The shape of Decoder's last-step state at layer 1 is: (64, 1024)
The shape of Decoder's last-step state at layer 2 is: (64, 1024)


### **5.3 The Full Translator Model**
We combine the Encoder and Decoder components into a subclassed Keras `Model`.


In [35]:
class Translator(Model):

    @classmethod
    def add_method(cls, fun):
      setattr(cls, fun.__name__, fun)
      return fun

    def __init__(self, context_text_processor, target_text_processor, target_idx_to_word,
                 embedding_dim, units, num_layers, **kwargs):
        super(Translator, self).__init__(**kwargs)
        # Build the encoder and decoder
        self.encoder  = Encoder(context_text_processor,
                                embedding_dim, units, num_layers,
                                name="seq2seq_encoder")
        self.decoder = Decoder(target_text_processor, target_idx_to_word,
                               embedding_dim, units, num_layers,
                               name="seq2seq_decoder")


    def call(self, input):
        context, x = input
        output, state = self.encoder(context)
        logits, state = self.decoder(state[-1][0], x, state)

        try:
            del logits._keras_mask    # delete the keras mask, so keras doesn't scale the loss+accuracy.
        except AttributeError: pass

        return logits

In [37]:
model = Translator(context_text_processor, target_text_processor, eng_idx_to_word,
                   embedding_dim, units, num_layers, name="seq2seq_translator")

logits = model((context, target_in))

print(f'The shape of the model output is: {logits.shape}')

The shape of the model output is: (64, 15, 5852)


In [38]:
model.summary()


Model: "seq2seq_translator"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ seq2seq_encoder (Encoder)       │ ?                      │    11,091,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ seq2seq_decoder (Decoder)       │ ?                      │    20,878,044 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,969,244 (121.95 MB)

 Trainable params: 31,969,244 (121.95 MB)

 Non-trainable params: 0 (0.00 B)

In [40]:
print("=== ENCODER SUMMARY ===")
model.encoder.summary()
print("\n=== DECODER SUMMARY ===")
model.decoder.summary()


=== ENCODER SUMMARY ===


Model: "seq2seq_encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (5, None)              │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_4 (Embedding)         │ (64, 20, 256)          │       855,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rnn_4 (RNN)                     │ ((64, 1024), (64,      │    10,235,904 │
│                                 │ 1024), (64, 1024))     │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,091,200 (42.31 MB)

 Trainable params: 11,091,200 (42.31 MB)

 Non-trainable params: 0 (0.00 B)

... (2 outputs left out)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ (5, None)              │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ string_lookup_3 (StringLookup)  │ (5, None)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_5 (Embedding)         │ (64, 15, 256)          │     1,498,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rnn_5 (RNN)                     │ ((64, 15, 1024), (64,  │    13,381,632 │
│                                 │ 1024), (64, 1024))     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (64, 15, 5852)         │     5,998,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ concatenate_2 (Concatenate)     │ (64, 15, 1280)         │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,878,044 (79.64 MB)

 Trainable params: 20,878,044 (79.64 MB)

 Non-trainable params: 0 (0.00 B)

### **Reflection Question 10.2**

(a) Why do we pass the final hidden state of the Encoder to initialize the Decoder's hidden state? How does this bridge the gap between the source and target languages?

(b) In both Encoder and Decoder, we set `mask_zero=True` in our `Embedding` layers. Explain how this zero-masking mechanism prevents the model from being confused by padding zeros.

**Answer:**
*Write your answer here (1-3 sentences per part).*


**Observe**: Jump to Step 7 and 8 to run the translation code on the sample sentences before model training. What do you see as translation outputs? Why are they structured this way?

# **Step 6: Model Training**

We will define custom loss and accuracy functions that ignore padding (`0` index) so that padding steps do not influence model parameters or performance metrics.


In [41]:
def masked_loss(y_true, y_pred):
    loss_obj = keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')
    loss = loss_obj(y_true, y_pred)
    mask = tf.cast(y_true != 0, loss.dtype)
    loss *= mask
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

def masked_acc(y_true, y_pred):
    y_pred = tf.argmax(y_pred, axis=-1)
    match = tf.cast(y_true == y_pred, tf.float32)
    mask = tf.cast(y_true != 0, tf.float32)
    match *= mask
    return tf.reduce_sum(match) / tf.reduce_sum(mask)


Since training a sequence-to-sequence model from scratch is a bit time-consuming, you can load pre-trained weights for a warm start:


In [42]:
model.compile(optimizer='adam', loss=masked_loss, metrics=[masked_acc, masked_loss])


In [45]:
model.load_weights("translator_seq2seq.weights.h5")


In [46]:
history = model.fit(train_ds, epochs=5, validation_data=val_ds)


Epoch 1/5
286/286 ━━━━━━━━━━━━━━━━━━━━ 74s 258ms/step - loss: 0.9315 - masked_acc: 0.7655 - masked_loss: 0.9324 - val_loss: 2.6282 - val_masked_acc: 0.5651 - val_masked_loss: 2.6259
Epoch 2/5
286/286 ━━━━━━━━━━━━━━━━━━━━ 74s 259ms/step - loss: 0.6313 - masked_acc: 0.8361 - masked_loss: 0.6312 - val_loss: 2.6470 - val_masked_acc: 0.5745 - val_masked_loss: 2.6467
Epoch 3/5
286/286 ━━━━━━━━━━━━━━━━━━━━ 74s 257ms/step - loss: 0.4060 - masked_acc: 0.8978 - masked_loss: 0.4059 - val_loss: 2.7007 - val_masked_acc: 0.5766 - val_masked_loss: 2.7042
Epoch 4/5
286/286 ━━━━━━━━━━━━━━━━━━━━ 83s 259ms/step - loss: 0.2561 - masked_acc: 0.9409 - masked_loss: 0.2562 - val_loss: 2.7574 - val_masked_acc: 0.5831 - val_masked_loss: 2.7563
Epoch 5/5
286/286 ━━━━━━━━━━━━━━━━━━━━ 74s 257ms/step - loss: 0.1699 - masked_acc: 0.9661 - masked_loss: 0.1697 - val_loss: 2.8326 - val_masked_acc: 0.5822 - val_masked_loss: 2.8339


### **Reflection Question 10.3**

(a) Describe how Keras leverages "Teacher Forcing" during the training of the Decoder. How does `target_in` differ from `target_out` to facilitate this?

(b) Why are standard categorical cross-entropy loss functions unsuitable for variable-length padded sequences without modification? Explain how `masked_loss` solves this problem.

**Answer:**
*Write your answer here (1-3 sentences per part).*


# **Step 7: Translation Inference**

For translating new sentences at inference time, we do not have target labels (true outputs). Instead, we feed the Decoder's predicted token from each step back as its input for the next step.


In [47]:
@Translator.add_method
def translate(self, text, max_length=50):
    # text expects a Python string

    text = tf.convert_to_tensor(text)[tf.newaxis]
    context_tokens = self.encoder.text_processor(text).to_tensor()
    output, state = self.encoder(context_tokens)

    context = state[-1][0]

    # set up the loop inputs
    tokens = []

    start_token = 2  # the index of the token [START]
    end_token = 3    # the index of the token [END]

    # done = [[FALSE]]
    done = tf.zeros([1, 1], dtype=tf.bool)
    # initialze the target input sequence to a single start token
    # next_token = [[2]]
    next_token = tf.fill([1, 1], start_token)

    for _ in range(max_length):

        logits, state = self.decoder(context, next_token, state)
        # the shape of next_token is (1, 1)
        next_token = tf.argmax(logits, axis=-1)
        # if a sequence produces an end token, set it done
        done = done | (next_token == end_token)
        # Once a sequence is done, include the padding character instead of the special token [END] in the output
        next_token = tf.where(done, tf.constant(0, dtype=tf.int64), next_token)
        tokens.append(next_token)
        if tf.executing_eagerly() and tf.reduce_all(done):
            break

    # stack the lists of tokens; the shape of tokens is (1, t)
    tokens = tf.concat(tokens, axis=-1)

    words = self.decoder.idx_to_word(tokens)
    result = tf.strings.reduce_join(words, axis=-1, separator=' ')
    # remove the special tokens
    result = tf.strings.regex_replace(result, r'^ *\[START\] *', '')
    result = tf.strings.regex_replace(result, r' *\[END\] *$', '')

    return result

# **Step 8: Perform Translation and Interactive Widget**

Let's test our translator model on some sample Chinese sentences:


In [48]:
print("Translate '狗追兔子。'")
print(model.translate(tf.constant('狗追兔子。')).numpy()[0].decode())
print("\nTranslate '女孩在教室的角落啜泣。'")
print(model.translate(tf.constant('女孩在教室的角落啜泣。')).numpy()[0].decode())
print("\nTranslate '今天教室裡的學生比昨天還要多。'")
print(model.translate(tf.constant('今天教室裡的學生比昨天還要多。')).numpy()[0].decode())
print("\nTranslate '你在这里的工作是什么？'")
print(model.translate(tf.constant('你在这里的工作是什么？')).numpy()[0].decode())
print("\nTranslate '我可以借這本書麼？'")
print(model.translate(tf.constant('我可以借這本書麼？')).numpy()[0].decode())
print("\nTranslate '你今晚還要工作嗎？'")
print(model.translate(tf.constant('你今晚還要工作嗎？')).numpy()[0].decode())
print("\nTranslate '如果你不想去, 就不必去了。'")
print(model.translate(tf.constant('如果你不想去, 就不必去了。')).numpy()[0].decode())


Translate '狗追兔子。'
the dog chased the rabbit . 

Translate '女孩在教室的角落啜泣。'
the girl was sobbing in the corner of the schoolroom . 

Translate '今天教室裡的學生比昨天還要多。'
there are many more students in the classroom today than yesterday . 

Translate '你在这里的工作是什么？'
what ' s the new here for here ? 

Translate '我可以借這本書麼？'
can i do this book to ? 

Translate '你今晚還要工作嗎？'
are you going to work today ? 

Translate '如果你不想去, 就不必去了。'
if you don ' t want to go , you don ' t need to . 


### **Interactive Translation Widget**

Run the cell below to test your own custom Chinese sentences using our trained model!


In [49]:
import ipywidgets as widgets
from IPython.display import display

text_input = widgets.Text(
    value='今天天氣真好。',
    placeholder='Type Chinese here...',
    description='Chinese:',
    disabled=False
)
button = widgets.Button(description="Translate")
output = widgets.Output()

def on_button_clicked(b):
    with output:
        output.clear_output()
        chinese_str = text_input.value
        if chinese_str.strip() == "":
            print("Please enter a Chinese sentence!")
            return
        result_tensor = model.translate(tf.constant(chinese_str))
        english_str = result_tensor.numpy()[0].decode()
        print(f"Source: {chinese_str}")
        print(f"English: {english_str}")

button.on_click(on_button_clicked)
display(text_input, button, output)


Text(value='今天天氣真好。', description='Chinese:', placeholder='Type Chinese here...')

Button(description='Translate', style=ButtonStyle())

Output()